# Relatorio Payroll - Employment Situation Report

Notebook para geracao automatica do relatorio Word (.docx) do Nonfarm Payrolls.

**Fontes de dados:**
- Bloomberg Terminal (primario) / FRED API (fallback)
- Bureau of Labor Statistics (BLS): https://www.bls.gov/news.release/empsit.b.htm

**Output:** `output/informes_eventos/reports/payroll/Mesa de Investimentos - PAYROLL {data}.docx`

In [ ]:
# Configuracao do ambiente
from pathlib import Path

from reports._paths import INPUT, OUTPUT

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from reports._style import COPOM, DPI, plot_config, clean_axes

plot_config(6)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# ============================================
# CONFIGURACAO
# ============================================
PLOT_START_DATE = '2022-01-01'   # Data inicial para graficos
REPORT_DATE = '2026-03-06'       # Data do release (YYYY-MM-DD)
# ============================================

In [ ]:
# ============================================
# USER INPUTS
# ============================================

# Valores de revisao (edite conforme necessario)
REVISAO_VALUES = {
    'Nonfarm Payrolls': None,              # e.g., '56k'
    'Unemployment Rate': None,              # e.g., '4.5%'
    'Average Hourly Earnings MoM': None,
    'Average Hourly Earnings YoY': None,
    'Labor Force Participation Rate': None,
}

# Texto resumo (deixe vazio para gerar automaticamente)
SUMMARY_TEXT = ""

# Screenshot de reacao do mercado (opcional - PNG fornecido pelo usuario)
MARKET_REACTION_PNG = None  # e.g., INPUT / 'market_reaction.png'

# Headlines Bloomberg (opcional - copie e cole)
HEADLINES_TEXT = """
"""

In [ ]:
# Imports dos modulos do payroll
from reports.payroll.core.data_loader import (
    fetch_payroll_data,
    fetch_extended_data,
    get_latest_release,
)
from reports.payroll.core.calculations import (
    calculate_moving_averages,
    format_release_summary,
    generate_summary_text,
    style_release_table,
)
from reports.payroll.core.bls_scraper import fetch_industry_breakdown
from reports.payroll.core.word_export import (
    WORD_DPI, WORD_FIGSIZE, WORD_FIGSIZE_DASHBOARD,
    WORD_TITLE_SIZE, WORD_LABEL_SIZE, WORD_TICK_SIZE, WORD_LEGEND_SIZE,
    create_styled_table_image, save_chart_for_word,
    create_unemployment_u6_chart, create_beveridge_curve,
)
from reports.payroll.core.word_report import generate_payroll_report

# Output directory
output_dir = OUTPUT / 'reports' / 'payroll'
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output: {output_dir}")

## 1. Dados do Release

In [ ]:
# Busca dados do release mais recente (Bloomberg)
release_data = None
release_summary = None

try:
    release_data = get_latest_release()
    release_summary = format_release_summary(release_data)
    display(style_release_table(release_summary))
except RuntimeError as e:
    print(f"Bloomberg nao disponivel: {e}")
    print("O release summary sera preenchido manualmente ou omitido do Word.")

In [ ]:
# Aplica revisoes e gera tabela estilizada para Word
if release_summary is not None and len(release_summary) > 0:
    for idx, row in release_summary.iterrows():
        evento = row['Evento']
        if evento in REVISAO_VALUES and REVISAO_VALUES[evento] is not None:
            release_summary.at[idx, 'Revisao'] = REVISAO_VALUES[evento]

    create_styled_table_image(
        data=release_summary,
        output_path=output_dir / 'release_summary_table_word.png',
        title='Payroll',
        highlight_column='Atual',
        col_widths=[2.5, 0.8, 1.0, 0.8, 1.0, 0.8],
    )
    print("Tabela estilizada salva.")

## 2. Serie Historica - Nonfarm Payrolls

In [ ]:
# Busca dados historicos (desde 1990) - Bloomberg ou FRED fallback
payroll_data = fetch_payroll_data(start_date='1990-01-01')
payroll_with_ma = calculate_moving_averages(payroll_data, column='NFP', windows=[3, 6])

print(f"Dados carregados: {len(payroll_data)} meses")
print(f"Periodo: {payroll_data.index[0].strftime('%Y-%m')} a {payroll_data.index[-1].strftime('%Y-%m')}")

display_cols = ['NFP', 'NFP_MA3m', 'NFP_MA6m', 'Unemployment', 'AHE_YoY']
available_cols = [c for c in display_cols if c in payroll_with_ma.columns]
payroll_with_ma[available_cols].tail(12).style.format('{:.1f}').set_caption('Ultimos 12 Meses')

In [ ]:
# Grafico: NFP com Medias Moveis (filtrado por PLOT_START_DATE)
plot_data = payroll_with_ma[payroll_with_ma.index >= pd.to_datetime(PLOT_START_DATE)].copy()

fig, ax = plt.subplots(figsize=WORD_FIGSIZE, dpi=WORD_DPI)

colors = [COPOM[1] if v >= 0 else COPOM[5] for v in plot_data['NFP']]
ax.bar(plot_data.index, plot_data['NFP'],
       color=colors, alpha=0.7, label='NFP Mensal', width=15)
ax.plot(plot_data.index, plot_data['NFP_MA3m'],
        color=COPOM[0], linewidth=2, label='Media 3m')
ax.plot(plot_data.index, plot_data['NFP_MA6m'],
        color=COPOM[2], linewidth=2, linestyle='--', label='Media 6m')
ax.axhline(y=0, color='black', linewidth=0.8)

ax.set_title('Nonfarm Payrolls - Variacao Mensal (mil)',
             fontsize=WORD_TITLE_SIZE, fontweight='bold')
ax.set_ylabel('Empregos (mil)', fontsize=WORD_LABEL_SIZE)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
ax.tick_params(axis='both', labelsize=WORD_TICK_SIZE)
ax.legend(loc='upper right', fontsize=WORD_LEGEND_SIZE)
clean_axes(ax)
plt.tight_layout()

save_chart_for_word(fig, output_dir / 'nfp_time_series_word.png')
plt.savefig(output_dir / 'nfp_time_series.png', dpi=DPI, bbox_inches='tight', facecolor='white')
plt.show()

## 3. Dashboard do Mercado de Trabalho

In [ ]:
# Dashboard 4 paineis (filtrado por PLOT_START_DATE)
plot_data = payroll_data[payroll_data.index >= pd.to_datetime(PLOT_START_DATE)].copy()

fig, axes = plt.subplots(2, 2, figsize=WORD_FIGSIZE_DASHBOARD, dpi=WORD_DPI)

# 1. Taxa de Desemprego
ax1 = axes[0, 0]
if 'Unemployment' in plot_data.columns:
    ax1.plot(plot_data.index, plot_data['Unemployment'], color=COPOM[0], linewidth=2)
ax1.set_title('Taxa de Desemprego (%)', fontsize=WORD_LABEL_SIZE, fontweight='bold')
ax1.set_ylabel('%', fontsize=WORD_TICK_SIZE)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
ax1.tick_params(axis='both', labelsize=WORD_TICK_SIZE - 1)
clean_axes(ax1)

# 2. Participacao na Forca de Trabalho
ax2 = axes[0, 1]
if 'LFPR' in plot_data.columns:
    ax2.plot(plot_data.index, plot_data['LFPR'], color=COPOM[2], linewidth=2)
ax2.set_title('Participacao na Forca de Trabalho (%)', fontsize=WORD_LABEL_SIZE, fontweight='bold')
ax2.set_ylabel('%', fontsize=WORD_TICK_SIZE)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
ax2.tick_params(axis='both', labelsize=WORD_TICK_SIZE - 1)
clean_axes(ax2)

# 3. Salarios YoY
ax3 = axes[1, 0]
if 'AHE_YoY' in plot_data.columns:
    ax3.plot(plot_data.index, plot_data['AHE_YoY'], color=COPOM[1], linewidth=2)
ax3.set_title('Salarios - Variacao Anual (%)', fontsize=WORD_LABEL_SIZE, fontweight='bold')
ax3.set_ylabel('% YoY', fontsize=WORD_TICK_SIZE)
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
ax3.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
ax3.tick_params(axis='both', labelsize=WORD_TICK_SIZE - 1)
clean_axes(ax3)

# 4. Salarios MoM
ax4 = axes[1, 1]
if 'AHE_MoM' in plot_data.columns:
    colors = [COPOM[1] if v >= 0.3 else COPOM[9] for v in plot_data['AHE_MoM']]
    ax4.bar(plot_data.index, plot_data['AHE_MoM'], color=colors, alpha=0.8, width=15)
    ax4.axhline(y=0.3, color=COPOM[5], linewidth=1.5, linestyle='--', label='Meta Fed (~0.3%)')
ax4.set_title('Salarios - Variacao Mensal (%)', fontsize=WORD_LABEL_SIZE, fontweight='bold')
ax4.set_ylabel('% MoM', fontsize=WORD_TICK_SIZE)
ax4.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
ax4.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
ax4.tick_params(axis='both', labelsize=WORD_TICK_SIZE - 1)
ax4.legend(loc='upper right', fontsize=WORD_TICK_SIZE - 1)
clean_axes(ax4)

plt.tight_layout()
save_chart_for_word(fig, output_dir / 'labor_market_dashboard_word.png')
plt.savefig(output_dir / 'labor_market_dashboard.png', dpi=DPI, bbox_inches='tight', facecolor='white')
plt.show()

## 4. Emprego por Setor

In [ ]:
# Busca dados do BLS
industry_data = fetch_industry_breakdown()
print(f"Dados do BLS: {len(industry_data)} setores")
display(industry_data.style.format({'current_month': '{:.0f}', 'prior_month': '{:.0f}', 'prior_year': '{:.0f}'})
        .set_caption('Emprego por Setor (mil)'))

In [ ]:
# Grafico: Barras horizontais por setor
INDUSTRY_ORDER = [
    'Total nonfarm', 'Total private', 'Goods-producing',
    'Mining and logging', 'Construction', 'Manufacturing',
    'Durable goods', 'Motor vehicles and parts', 'Nondurable goods',
    'Private service-providing', 'Wholesale trade', 'Retail trade',
    'Transportation and warehousing', 'Utilities', 'Information',
    'Financial activities', 'Professional and business services',
    'Temporary help services', 'Private education and health services',
    'Health care and social assistance', 'Leisure and hospitality',
]

plot_ind = industry_data[industry_data['industry'].isin(INDUSTRY_ORDER)].copy()
plot_ind = plot_ind.dropna(subset=['current_month'])

industry_cat = pd.CategoricalDtype(categories=INDUSTRY_ORDER[::-1], ordered=True)
plot_ind['industry'] = plot_ind['industry'].astype(industry_cat)
plot_ind = plot_ind.sort_values('industry')

if len(plot_ind) > 0:
    n_sectors = len(plot_ind)
    fig_height = max(6.0, n_sectors * 0.42)
    fig, ax = plt.subplots(figsize=(10.0, fig_height), dpi=WORD_DPI)
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')

    colors = [COPOM[1] if v >= 0 else '#D4A0A0' for v in plot_ind['current_month']]
    y_pos = range(len(plot_ind))
    ax.barh(y_pos, plot_ind['current_month'], color=colors, alpha=0.75, height=0.65)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(plot_ind['industry'], fontsize=11)
    ax.tick_params(axis='y', length=0)
    ax.spines['left'].set_visible(False)
    ax.axvline(x=0, color='black', linewidth=0.8, zorder=2)

    x_range = plot_ind['current_month'].max() - plot_ind['current_month'].min()
    for i, (idx, row) in enumerate(plot_ind.iterrows()):
        val = row['current_month']
        offset = max(1.0, x_range * 0.015) if val >= 0 else -max(1.0, x_range * 0.015)
        ha = 'left' if val >= 0 else 'right'
        ax.annotate(f'{val:.0f}', xy=(val + offset, i), va='center', ha=ha, fontsize=11, color='#333333')

    ax.set_title('Variacao do Emprego por Setor (mil)', fontsize=18, fontweight='bold', pad=15)
    ax.set_xlabel('Variacao (mil)', fontsize=13)
    ax.tick_params(axis='x', labelsize=11)
    clean_axes(ax)
    plt.tight_layout(pad=1.5)

    save_chart_for_word(fig, output_dir / 'industry_breakdown_word.png')
    plt.savefig(output_dir / 'industry_breakdown.png', dpi=DPI, bbox_inches='tight', facecolor='white')
    plt.show()

## 5. Indicadores Complementares (U6, JOLTS)

In [ ]:
# Busca dados estendidos (U6, JOLTS) do FRED
extended_data = None
try:
    extended_data = fetch_extended_data(start_date='1990-01-01')
    print(f"Dados estendidos: {len(extended_data)} meses")
    print(f"Colunas: {list(extended_data.columns)}")
    extended_data.tail()
except Exception as e:
    print(f"Dados estendidos nao disponiveis: {e}")

In [ ]:
# Graficos U6 e Beveridge (se dados disponiveis)
if extended_data is not None:
    # U3 vs U6
    if 'U6' in extended_data.columns and 'Unemployment' in payroll_data.columns:
        create_unemployment_u6_chart(
            payroll_data, extended_data,
            output_dir / 'u6_chart_word.png',
            plot_start_date=PLOT_START_DATE,
        )
        print("Grafico U3 vs U6 salvo.")

    # Beveridge Curve
    if 'JOLTS' in extended_data.columns and 'Unemployment' in payroll_data.columns:
        # Alinhar indices
        merged = payroll_data[['Unemployment']].join(extended_data[['JOLTS']], how='inner').dropna()
        if len(merged) > 0:
            create_beveridge_curve(
                merged['Unemployment'], merged['JOLTS'],
                output_dir / 'beveridge_curve_word.png',
                plot_start_date=PLOT_START_DATE,
            )
            print("Curva de Beveridge salva.")

## 6. Gerar Documento Word

In [ ]:
# Monta chart_paths para o Word report
chart_paths = {}

nfp_path = output_dir / 'nfp_time_series_word.png'
if nfp_path.exists():
    chart_paths['nfp_chart'] = nfp_path

dashboard_path = output_dir / 'labor_market_dashboard_word.png'
if dashboard_path.exists():
    chart_paths['dashboard'] = dashboard_path

industry_path = output_dir / 'industry_breakdown_word.png'
if industry_path.exists():
    chart_paths['industry'] = industry_path

u6_path = output_dir / 'u6_chart_word.png'
if u6_path.exists():
    chart_paths['u6_chart'] = u6_path

beveridge_path = output_dir / 'beveridge_curve_word.png'
if beveridge_path.exists():
    chart_paths['beveridge'] = beveridge_path

print(f"Charts disponiveis: {list(chart_paths.keys())}")

# Gera texto resumo (automatico ou manual)
summary = SUMMARY_TEXT
if not summary and release_data is not None:
    summary = generate_summary_text(release_data, industry_data)
    print(f"\nResumo gerado automaticamente:\n{summary}")

# Gera release_summary placeholder se Bloomberg nao disponivel
if release_summary is None:
    release_summary = pd.DataFrame({
        'Evento': ['Nonfarm Payrolls', 'Unemployment Rate', 'Average Hourly Earnings MoM',
                    'Average Hourly Earnings YoY', 'Labor Force Participation Rate'],
        'Periodo': ['-'] * 5,
        'Pesquisa': ['-'] * 5,
        'Atual': ['-'] * 5,
        'Anterior': ['-'] * 5,
        'Revisao': [''] * 5,
    })
    print("\nRelease summary: usando placeholder (Bloomberg indisponivel)")

In [ ]:
# Gera o documento Word
market_path = Path(MARKET_REACTION_PNG) if MARKET_REACTION_PNG else None

docx_path = generate_payroll_report(
    release_date=REPORT_DATE,
    release_summary=release_summary,
    chart_paths=chart_paths,
    summary_text=summary,
    headlines_text=HEADLINES_TEXT,
    market_reaction_path=market_path,
    output_dir=output_dir,
)

print(f"\nDocumento Word gerado: {docx_path}")
print(f"Tamanho: {docx_path.stat().st_size / 1024:.0f} KB")

## 7. Exportar Dados

In [ ]:
# Exporta dados para Excel
report_date_str = REPORT_DATE.replace('-', '')
excel_path = output_dir / f'payroll_analysis_{report_date_str}.xlsx'

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    payroll_with_ma.to_excel(writer, sheet_name='Serie Historica')

    if industry_data is not None and len(industry_data) > 0:
        industry_data.to_excel(writer, sheet_name='Breakdown Setorial', index=False)

    if release_summary is not None:
        release_summary.to_excel(writer, sheet_name='Release Atual', index=False)

    if extended_data is not None:
        extended_data.to_excel(writer, sheet_name='Indicadores Estendidos')

print(f"Excel exportado: {excel_path}")

---

**Notas:**
- Dados: Bloomberg (primario) com fallback automatico para FRED API
- Industry breakdown: BLS Table B (web scraper)
- U6 e JOLTS: FRED API
- Todos os graficos e o `.docx` sao salvos em `output/informes_eventos/reports/payroll/`
- Altere `REPORT_DATE` e `PLOT_START_DATE` no inicio do notebook para cada release